In [ ]:
import oracledb
import ssl
import pandas as pd

# === DADOS DE CONEXÃO === #
usuario = "BRUNOBRIANI_SCHEMA_8DADJ"
senha = "5RHMY4#9QDbPGMOBOLDXUXAK8WJ2WI"
dsn = "(description=(address=(protocol=tcps)(port=2484)(host=db.freesql.com))(connect_data=(service_name=23ai_34ui2)))"

# === Configuração SSL === #
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

try:
    # === Conexão === #
    connection = oracledb.connect(user=usuario, password=senha, dsn=dsn, ssl_context=ctx)
    print("Conectado com sucesso ao banco!")

    # === QUERYS SQL === #
    query01 = """
    SELECT 
        e.first_name, 
        e.last_name, 
        e.hire_date, 
        j.job_title, 
        d.department_name, 
        e.salary
    FROM HR.employees e
    LEFT JOIN HR.jobs j ON e.job_id = j.job_id
    LEFT JOIN HR.departments d ON e.department_id = d.department_id
    WHERE d.department_name IS NOT NULL 
      AND e.salary > 2800
    ORDER BY d.department_name, j.job_title, e.salary DESC
    """  

    query02 = """
    SELECT 
        e.employee_id,
        e.first_name,
        e.last_name,
        e.salary,
        e.hire_date, 
        d.department_name,
        l.city,
        c.country_name,
        r.region_name
    FROM HR.employees e
    LEFT JOIN HR.departments d ON e.department_id = d.department_id
    LEFT JOIN HR.locations l ON d.location_id = l.location_id
    LEFT JOIN HR.countries c ON l.country_id = c.country_id
    LEFT JOIN HR.regions r ON c.region_id = r.region_id
    WHERE r.region_name IS NOT NULL AND e.salary > 2800
    ORDER BY r.region_name, c.country_name, d.department_name, e.salary DESC
    """  

    # === Carrega os dados === #
    df1 = pd.read_sql(query01, connection)
    df2 = pd.read_sql(query02, connection)

    print("\n--- DADOS QUERY 1 ---")
    print(df1.head())
    
    print("\n--- DADOS QUERY 2 ---")
    print(df2.head())

    # === Exportar para a pasta criada (Dados) === #
    df1.to_csv("Dados/query_01.csv", index=False)
    df2.to_csv("Dados/query_02.csv", index=False)

    connection.close()
    print("\nConexão encerrada e arquivos CSV gerados na pasta Dados.")

except Exception as e:
    print(f"\nErro na execução: {e}")


In [ ]:
# === Análise Exploratória de Dados (EDA) QUERY_01 === #
import pandas as pd

# === 1. Carrega os Dados da Query 01 === #
df_1 = pd.read_csv("Dados/query_01.csv")

# === 2. Diagnóstico Dos Dados === #
print("=== DIAGNÓSTICO DOS DADOS PARA TODAS AS VARIÁVEIS ===")

# === Verifica nulos em todas as colunas === #
nulos = df_1.isnull().sum()
print(f"\nValores Nulos por coluna:\n{nulos[nulos > 0] if nulos.sum() > 0 else '0'}")

# === Verifica duplicados em todas as linhas === #
duplicados = df_1.duplicated().sum()
print(f"\nLinhas duplicadas encontradas no conjunto de dados total: {duplicados}")

# === Transformação de data para datetime === #
df_1['HIRE_DATE'] = pd.to_datetime(df_1['HIRE_DATE'])
df_1['HIRE_YEAR'] = df_1['HIRE_DATE'].dt.year

print("\n=== Base 100% verificada e limpa ===")


In [ ]:
import numpy as np

# === Análise Exploratória Básica === #
print("=== VISÃO GERAL DOS DADOS (QUERY 01) ===")
print(df_func.info())

# === Criar a coluna de Ano para a análise temporal === #
df_func['HIRE_DATE'] = pd.to_datetime(df_func['HIRE_DATE'])
df_func['HIRE_YEAR'] = df_func['HIRE_DATE'].dt.year

# === Lista de métricas estatísticas Query 01 === #
metricas = ['mean', 'median', 'min', 'max', 'std', 'var']

In [ ]:
# === Cálcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Departamento' === #

print("\n=== ESTATÍSTICA SALARIAL POR DEPARTAMENTO ===")
estatistica_depto = df_func.groupby('DEPARTMENT_NAME')['SALARY'].agg(metricas)
estatistica_depto.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']
from tabulate import tabulate

# === Arredondar todos os valores para 2 casas decimais === #
estatistica_depto = estatistica_depto.round(2)

# === Exibir tabela com o tabulate === #
print(tabulate(estatistica_depto.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))

In [ ]:
# === Cálcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Cargo' === #

print("\n=== ESTATÍSTICA SALARIAL POR CARGO ===")
estatistica_cargo = df_func.groupby('JOB_TITLE')['SALARY'].agg(metricas)
estatistica_cargo.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']

# Arredonda todos os valores para 2 casas decimais
estatistica_cargo = estatistica_cargo.round(2)

# Agora exiba com o tabulate (ou print)
print(tabulate(estatistica_cargo.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))

In [ ]:
# Calcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Ano de Contratação'

print("\n=== ESTATÍSTICA SALARIAL POR ANO DE CONTRATAÇÃO ===")
estatistica_ano = df_func.groupby('HIRE_YEAR')['SALARY'].agg(metricas)
estatistica_ano.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']

# === Arredonda os valores para 2 casas decimais === #
estatistica_ano = estatistica_ano.round(2)

# === Ordena corretamente usando sort_values === #
print(tabulate(estatistica_ano.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# GRÁFICOS DE ANÁLISE SALARIAL POR DEPARTAMENTO #

# === Configuração de estilo === #
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Cria a figura com dois subplots ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# === Define a paleta de cores === #
paleta = sns.color_palette("viridis", n_colors=len(df_func['DEPARTMENT_NAME'].unique()))

# === GRÁFICO 1: Boxplot (Salário x Departamento) === #
sns.boxplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax1, 
    palette=paleta, 
    hue='DEPARTMENT_NAME', 
    legend=False,
    linewidth=1.2,
    fliersize=4
)
ax1.set_title('Distribuição Salarial por Departamento', fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Salário ($)')
ax1.set_ylabel('Departamento')

# === GRÁFICO 2: Média + Desvio Padrão (Estatística) === #
# = Design das Barras das médias com transparência = #
sns.barplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax2, 
    palette=paleta, 
    hue='DEPARTMENT_NAME', 
    alpha=0.4, 
    errorbar=None, 
    legend=False
)

# === Design das Linhas e pontos de Desvio Padrão === #
sns.pointplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax2, 
    color='black', 
    errorbar='sd', 
    join=False, 
    capsize=.15, 
    scale=0.6,                
    err_kws={'linewidth': 1.2} 
)

ax2.set_title('Média e Desvio Padrão', fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Salário Médio ($)')
ax2.set_ylabel('')

# === Ajuste final de layout === #
plt.tight_layout(pad=3.0)

# === Salva e exibe=== #
plt.savefig('analise_salarial_estatistica.png', dpi=300)
plt.show()

In [ ]:
# GRÁFICOS DE ANÁLISE SALARIAL POR CARGO #

# === Configuração de estilo === #
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Cria a figura  === #
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 12))

# === Define paleta de cores === #
paleta_cargo = sns.color_palette("viridis", n_colors=len(df_func['JOB_TITLE'].unique()))

# === GRÁFICO 1: Boxplot por Cargo === #
sns.boxplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax1, 
    palette=paleta_cargo, 
    hue='JOB_TITLE', 
    legend=False,
    linewidth=1.2,
    fliersize=4
)
ax1.set_title('Distribuição Salarial por Cargo', fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel('Salário ($)')
ax1.set_ylabel('Cargo')

# === GRÁFICO 2: Média + Desvio Padrão por Cargo === #
# Define design das Barras das médias
sns.barplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax2, 
    palette=paleta_cargo, 
    hue='JOB_TITLE', 
    alpha=0.4, 
    errorbar=None, 
    legend=False
)

# === Define design das Linhas de desvio padrão === #
sns.pointplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax2, 
    color='black', 
    errorbar='sd', 
    join=False, 
    capsize=.15, 
    scale=0.6,
    err_kws={'linewidth': 1.2}
)

ax2.set_title('Média e Desvio Padrão por Cargo', fontsize=18, fontweight='bold', pad=20)
ax2.set_xlabel('Salário Médio ($)')
ax2.set_ylabel('')

# === Ajuste final === #
plt.tight_layout(pad=3.0)

# === Salva e exibe === #
plt.savefig('analise_salarial_cargos.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# === Define a ordem dos anos do MAIOR para o MENOR === #
anos_invertidos = sorted(df_func['HIRE_YEAR'].unique(), reverse=True)

# === Configurações de estilo === #
sns.set_theme(style="white", context="paper")
plt.rcParams['font.family'] = 'sans-serif'
cor_base = "#88c999" 

corr, p_value = pearsonr(df_func['HIRE_YEAR'], df_func['SALARY'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# === GRÁFICO 1: Boxplot (Com ordem invertida) === #
sns.boxplot(data=df_func, x='HIRE_YEAR', y='SALARY', ax=ax1, 
            order=anos_invertidos,  # <--- ORDEM INVERTIDA AQUI
            color=cor_base, linewidth=0.8, fliersize=3,
            boxprops=dict(alpha=0.6, edgecolor='black', linewidth=0.8))
ax1.set_title('Distribuição Salarial (Ano mais recente p/ mais antigo)', fontsize=13, fontweight='bold', pad=15)
ax1.set_xlabel('Ano de Contratação')
ax1.set_ylabel('Salário (R$)')
sns.despine()

# === GRÁFICO 2: Tendência (Regressão invertida no eixo X) === #
sns.regplot(data=df_func, x='HIRE_YEAR', y='SALARY', ax=ax2, 
            scatter_kws={'alpha': 0.5, 'color': cor_base, 's': 30, 'edgecolor': 'white'}, 
            line_kws={'color': '#5a5a5a', 'linewidth': 1.5, 'linestyle': '--'})

# === Inverter o eixo X manualmente para o gráfico de dispersão === #
ax2.set_xlim(ax2.get_xlim()[::-1])

ax2.set_title(f'Tendência Temporal ($r = {corr:.2f}$)', fontsize=13, fontweight='bold', pad=15)
ax2.set_xlabel('Ano de Contratação (Decrescente)')
ax2.set_ylabel('')
sns.despine()

plt.tight_layout(pad=3.0)
plt.savefig('analise_temporal_invertida.png', dpi=300)
plt.show()